In [1]:
import sys
import os
import pandas as pd
import numpy as np
import scanpy as sc
import anndata
import matplotlib.pyplot as plt
from scipy.io import mmread
sys.executable

'/N/u/echimal/Quartz/.conda/envs/integration_env/bin/python'

In [2]:
#Data path
data_path = '/N/u/echimal/Quartz/Desktop/CLR_MRI/Human_GeoMx_Sep2025/'

In [3]:
##Load in Python as AnnData (Data derived from Seurat Analysis)
os.chdir(data_path)
# Load counts
counts = mmread("AD_CAA_counts.mtx").T.tocsr()  # transpose if necessary

# Load metadata
meta = pd.read_csv("AD_CAA_meta.csv", index_col=0)

# Load gene names
genes = pd.read_csv("AD_CAA_genes.csv")
gene_ids = genes['gene_id'].values

In [4]:
# Create AnnData
adata = anndata.AnnData(X=counts)
adata.var_names = gene_ids
adata.obs = meta

# Add counts as a layer
adata.layers["counts"] = adata.X.copy()  # raw counts layer

# Check
print(adata)
print(adata.layers.keys())
print(adata.obs.head())

AnnData object with n_obs × n_vars = 98653 × 35593
    obs: 'New_Idents', 'FDX'
    layers: 'counts'
KeysView(Layers with keys: counts)
                                   New_Idents     FDX
BRC2495_AAACCAAAGATAGGTA-1  Oligodendrocytes4  AD+CAA
BRC2495_AAACCAAAGCGGTACT-1  Oligodendrocytes4  AD+CAA
BRC2495_AAACCAAAGGAGGTGC-1  Oligodendrocytes1  AD+CAA
BRC2495_AAACCAAAGTATGAGA-1               OPC1  AD+CAA
BRC2495_AAACCAAAGTGGCTGT-1  Oligodendrocytes4  AD+CAA


In [5]:
adata.var = pd.DataFrame(index=gene_ids)
adata.var.index.name = "gene"
adata.obs["Experiment"] = "Batch1"
adata.obs.rename(columns={
    "New_Idents": "celltype"
}, inplace=True)

In [6]:
# quick checks
print(adata)
print(adata.obs.head())
print(adata.var.head())
print(adata.layers.keys())
print(adata.layers['counts'].dtype)
print(adata.layers['counts'].max(), adata.layers['counts'].min())

AnnData object with n_obs × n_vars = 98653 × 35593
    obs: 'celltype', 'FDX', 'Experiment'
    layers: 'counts'
                                     celltype     FDX Experiment
BRC2495_AAACCAAAGATAGGTA-1  Oligodendrocytes4  AD+CAA     Batch1
BRC2495_AAACCAAAGCGGTACT-1  Oligodendrocytes4  AD+CAA     Batch1
BRC2495_AAACCAAAGGAGGTGC-1  Oligodendrocytes1  AD+CAA     Batch1
BRC2495_AAACCAAAGTATGAGA-1               OPC1  AD+CAA     Batch1
BRC2495_AAACCAAAGTGGCTGT-1  Oligodendrocytes4  AD+CAA     Batch1
Empty DataFrame
Columns: []
Index: [DDX11L2, ENSG00000238009, ENSG00000239945, ENSG00000241860, ENSG00000286448]
KeysView(Layers with keys: counts)
float64
14344.927569186973 0.0


In [7]:
import scipy.sparse as sp
#Round (because of soupX the sparce matrix is float, and Negative Binomial need integers)
# make a copy to be safe
X = adata.layers['counts'].copy()

# if sparse
if sp.issparse(X):
    # remove any tiny negatives then round inplace
    X.data[X.data < 0] = 0
    X.data = np.rint(X.data)  # round to nearest integer
    X = X.astype('int64')     # convert dtype
else:
    X = np.array(X, copy=True)
    X[X < 0] = 0
    X = np.rint(X).astype('int64')

# replace both .layers and .X
adata.layers['counts'] = X
adata.X = X.copy()

In [8]:
#Filter low-quality cells and genes
sc.pp.filter_cells(adata, min_genes=1)
sc.pp.filter_genes(adata, min_cells=1)
print("After minimal filtering:", adata.shape)
X = adata.layers["counts"]  # sparse matrix or numpy array
# compute n_cells and non-zero mean
# careful with sparse matrices
if hasattr(X, "tocsc") or hasattr(X, "tocoo"):
    # sparse
    n_cells = (X > 0).sum(axis=0).A1
    total_counts = X.sum(axis=0).A1
else:
    n_cells = (X > 0).sum(axis=0)
    total_counts = X.sum(axis=0)

adata.var['n_cells'] = n_cells
# avoid division-by-zero:
adata.var['nonz_mean'] = np.divide(total_counts, np.where(n_cells==0, 1, n_cells))

nonz_mean_cutoff = np.log10(1.12)
cell_count_cutoff = np.log10(adata.shape[0] * 0.0005)
cell_count_cutoff2 = np.log10(adata.shape[0] * 0.03)

gene_mask = (
    (
        (np.log10(adata.var['nonz_mean']) > nonz_mean_cutoff) |
        (np.log10(adata.var['n_cells']) > cell_count_cutoff2)
    )
    &
    (np.log10(adata.var['n_cells']) > cell_count_cutoff)
)

print(f"Keeping {gene_mask.sum()} / {adata.n_vars} genes after gene filtering")
adata = adata[:, gene_mask].copy()
adata.raw = adata # sets .raw to point to current adata
# Keep a raw AnnData for later (cell2location expects use_raw=True often)

After minimal filtering: (98653, 35581)
Keeping 21347 / 35581 genes after gene filtering


In [9]:
# quick checks
print(adata)
print(adata.obs.head())
print(adata.var.head())
print(adata.layers.keys())
print(adata.layers['counts'].dtype)
print(adata.layers['counts'].max(), adata.layers['counts'].min())

AnnData object with n_obs × n_vars = 98653 × 21347
    obs: 'celltype', 'FDX', 'Experiment', 'n_genes'
    var: 'n_cells', 'nonz_mean'
    layers: 'counts'
                                     celltype     FDX Experiment  n_genes
BRC2495_AAACCAAAGATAGGTA-1  Oligodendrocytes4  AD+CAA     Batch1     4435
BRC2495_AAACCAAAGCGGTACT-1  Oligodendrocytes4  AD+CAA     Batch1     3409
BRC2495_AAACCAAAGGAGGTGC-1  Oligodendrocytes1  AD+CAA     Batch1     4180
BRC2495_AAACCAAAGTATGAGA-1               OPC1  AD+CAA     Batch1     3905
BRC2495_AAACCAAAGTGGCTGT-1  Oligodendrocytes4  AD+CAA     Batch1     2555
                 n_cells  nonz_mean
gene                               
ENSG00000241860     7135   1.128662
ENSG00000290385     3130   1.053035
ENSG00000291215     8133   1.101316
LINC01409          28120   2.070555
ENSG00000290784     9733   1.190691
KeysView(Layers with keys: counts)
int64
14345 0


In [30]:
import scvi
from cell2location.models import RegressionModel
import cell2location
scvi.settings.seed = 0
import torch #This Part need top be run in a session with GPU Otherwise the net blocks will fail (or take more than 24 hours)
print(torch.cuda.is_available())  # Should return True
print(torch.cuda.device_count())   # Should return 1 or more

[rank: 0] Seed set to 0


True
1


In [10]:
#Path to Regression results
rresults = '/N/u/echimal/Quartz/Desktop/CLR_MRI/Human_GeoMx_Sep2025/Regression-model/'

In [11]:
import lightning.fabric.plugins.environments.mpi as lightning_mpi

def _no_mpi_detect():
    return False

# Disable MPI detection
lightning_mpi.MPIEnvironment.detect = staticmethod(_no_mpi_detect)


In [31]:
# List of conditions
fdx_conditions = ['AD+CAA', 'Control']

# Dictionary to store trained models
trained_models = {}

for fdx in fdx_conditions:
    print(f"Training RegressionModel for {fdx}...")
    
    # Subset the AnnData to only this condition
    adata_sub = adata[adata.obs['FDX'] == fdx].copy()
    
    # Setup the AnnData for cell2location
    RegressionModel.setup_anndata(
        adata_sub,
        batch_key='Experiment',
        labels_key='celltype',
        layer='counts'
    )
    
    # Create the RegressionModel
    mod_sub = RegressionModel(adata_sub)
    
    # Train the model
    mod_sub.train(max_epochs=100, batch_size=1024, lr=0.01, accelerator='gpu')
    
    # Save the model in the dictionary
    trained_models[fdx] = mod_sub
#mod.train(max_epochs=100, batch_size=1024, lr=0.01, use_gpu=True) #GPU avail

Training RegressionModel for AD+CAA...


/N/u/echimal/Quartz/.conda/envs/integration_env/lib/python3.10/site-packages/scvi/data/fields/_dataframe_field.py:187: UserWarning: Category 27 in adata.obs['_scvi_labels'] has fewer than 3 cells. Models may not train properly.
  categorical_mapping = _make_column_categorical(
/N/u/echimal/Quartz/.conda/envs/integration_env/lib/python3.10/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /N/u/echimal/Quartz/.conda/envs/integration_env/lib/ ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/N/u/echimal/Quartz/.conda/envs/integration_env/lib/python3.10/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command wi

Training:   0%|          | 0/100 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=100` reached.


Training RegressionModel for Control...


/N/u/echimal/Quartz/.conda/envs/integration_env/lib/python3.10/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /N/u/echimal/Quartz/.conda/envs/integration_env/lib/ ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
/N/u/echimal/Quartz/.conda/envs/integration_env/lib/python3.10/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /N/u/echimal/Quartz/.conda/envs/integration_env/lib/ ...
/N/u/echimal/Quartz/.conda/envs/integration_env/lib/python3.10/site-packages/lightning/pytorch/trainer/configuration_validator.py:68: You passed in a `val_dataloader` but have no `validation

Training:   0%|          | 0/100 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=100` reached.


In [33]:
for fdx, model in trained_models.items():
    model.save(f"{rresults}/{fdx}_regression_model/")

In [17]:
#After Training and saving the models you can run : 
from cell2location.models import RegressionModel

# Subset again for loading if you want per-condition models
adata_AD = adata[adata.obs['FDX'] == 'AD+CAA'].copy()
adata_CTRL = adata[adata.obs['FDX'] == 'Control'].copy()

mod_AD = RegressionModel.load(f"{rresults}/AD+CAA_regression_model/", adata=adata_AD)
mod_AD.export_posterior(
    adata_AD,
    sample_kwargs={"num_samples": 1000, "batch_size": 2500}
)

mod_CTRL = RegressionModel.load(f"{rresults}/Control_regression_model/", adata=adata_CTRL)
mod_CTRL.export_posterior(
    adata_CTRL,
    sample_kwargs={"num_samples": 1000, "batch_size": 2500}
)



INFO     File                                                                                                      
         /N/u/echimal/Quartz/Desktop/CLR_MRI/Human_GeoMx_Sep2025/Regression-model//AD+CAA_regression_model/model.pt
         already downloaded                                                                                        


/N/u/echimal/Quartz/.conda/envs/integration_env/lib/python3.10/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /N/u/echimal/Quartz/.conda/envs/integration_env/lib/ ...
/N/u/echimal/Quartz/.conda/envs/integration_env/lib/python3.10/site-packages/scvi/data/fields/_dataframe_field.py:227: UserWarning: Category 27 in adata.obs['_scvi_labels'] has fewer than 3 cells. Models may not train properly.
  new_mapping = _make_column_categorical(
/N/u/echimal/Quartz/.conda/envs/integration_env/lib/python3.10/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /N/u/echimal/Quartz/.conda/envs/integration_env/lib/ ...


Training:   0%|          | 0/368 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=1` reached.
/N/u/echimal/Quartz/.conda/envs/integration_env/lib/python3.10/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /N/u/echimal/Quartz/.conda/envs/integration_env/lib/ ...


Sampling local variables, batch:   0%|          | 0/22 [00:00<?, ?it/s]

...

Training:   0%|          | 0/451 [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=1` reached.
/N/u/echimal/Quartz/.conda/envs/integration_env/lib/python3.10/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /N/u/echimal/Quartz/.conda/envs/integration_env/lib/ ...


Sampling local variables, batch:   0%|          | 0/18 [00:00<?, ?it/s]

Sampling global variables, sample:   0%|          | 0/999 [00:00<?, ?it/s]

AnnData object with n_obs × n_vars = 44365 × 21347
    obs: 'celltype', 'FDX', 'Experiment', 'n_genes', '_indices', '_scvi_batch', '_scvi_labels'
    var: 'n_cells', 'nonz_mean'
    uns: '_scvi_uuid', '_scvi_manager_uuid', 'mod'
    varm: 'means_per_cluster_mu_fg', 'stds_per_cluster_mu_fg', 'q05_per_cluster_mu_fg', 'q95_per_cluster_mu_fg'
    layers: 'counts'

In [18]:
adata_AD.uns['mod']['post_sample_means']['per_cluster_mu_fg'] #Must be populated or there is a problem 


array([[4.96029593e-02, 4.27584462e-02, 1.52064011e-01, ...,
        9.10247269e-04, 5.41759133e-01, 1.56063708e-02],
       [2.91869864e-02, 5.02038635e-02, 1.17670782e-01, ...,
        4.33453545e-03, 8.44928265e-01, 4.80438083e-01],
       [6.45307228e-02, 4.91913222e-02, 1.65385634e-01, ...,
        2.28812685e-03, 5.46805918e-01, 3.32000017e-01],
       ...,
       [6.44081756e-02, 3.09723597e-02, 5.82264587e-02, ...,
        2.26098523e-01, 1.00240004e+00, 1.13366210e+00],
       [3.65543403e-02, 5.48518226e-02, 6.15510046e-02, ...,
        8.60202536e-02, 9.85521913e-01, 9.34099495e-01],
       [1.04414895e-01, 5.85332774e-02, 1.70584500e-01, ...,
        4.77074236e-01, 1.07983983e+00, 1.11660886e+00]],
      shape=(46, 21347), dtype=float32)

In [23]:
import matplotlib.pyplot as plt
import matplotlib as mpl
from re import sub
import os
import cell2location

def export_and_compare_signatures(adata, covariate_col="celltype", model_name="model", output_folder="./"):
    os.makedirs(output_folder, exist_ok=True)

    # 1. Extract inferred signatures (v0.1.5 location)
    mu = adata.uns['mod']['post_sample_means']['per_cluster_mu_fg']
    
    # mu shape is (n_celltypes, n_genes)
    # We want (n_genes, n_celltypes)
    mu = mu.T  

    # Correct column order: sorted unique cluster labels
    celltypes = sorted(adata.obs[covariate_col].unique())
    # Convert to dataframe
    inferred_df = pd.DataFrame(
        mu,
        index=adata.var_names,
        columns=adata.obs[covariate_col].unique()
    )
    
    inferred_df.to_csv(f"{output_folder}/{model_name}_inferred_signatures.csv")
    print(f"Saved inferred signatures -> {output_folder}/{model_name}_inferred_signatures.csv")

    # 2. Compute analytical cluster averages directly from counts
    aver_df = (
        adata.to_df()    # gene x cell
        .join(adata.obs[covariate_col])
        .groupby(covariate_col)
        .mean()
        .T               # gene x celltype
    )

    aver_df.to_csv(f"{output_folder}/{model_name}_mean_cluster_expr.csv")
    print(f"Saved analytical averages -> {output_folder}/{model_name}_mean_cluster_expr.csv")

    return inferred_df, aver_df

In [24]:
# AD+CAA
inf_AD, aver_AD = export_and_compare_signatures(adata_AD, covariate_col='celltype', model_name='AD+CAA', output_folder=rresults)

# Control
inf_CTRL, aver_CTRL = export_and_compare_signatures(adata_CTRL, covariate_col='celltype', model_name='Control', output_folder=rresults)


Saved inferred signatures -> /N/u/echimal/Quartz/Desktop/CLR_MRI/Human_GeoMx_Sep2025/Regression-model//AD+CAA_inferred_signatures.csv
Saved analytical averages -> /N/u/echimal/Quartz/Desktop/CLR_MRI/Human_GeoMx_Sep2025/Regression-model//AD+CAA_mean_cluster_expr.csv
Saved inferred signatures -> /N/u/echimal/Quartz/Desktop/CLR_MRI/Human_GeoMx_Sep2025/Regression-model//Control_inferred_signatures.csv
Saved analytical averages -> /N/u/echimal/Quartz/Desktop/CLR_MRI/Human_GeoMx_Sep2025/Regression-model//Control_mean_cluster_expr.csv


In [25]:
import sys
for module in sys.modules:
    try:
        print(module,sys.modules[module].__version__)
    except:
        try:
            if  type(sys.modules[module].version) is str:
                print(module,sys.modules[module].version)
            else:
                print(module,sys.modules[module].version())
        except:
            try:
                print(module,sys.modules[module].VERSION)
            except:
                pass

sys 3.10.19 | packaged by conda-forge | (main, Oct 22 2025, 22:29:10) [GCC 14.3.0]
re 2.2.1
ipaddress 1.0
ipykernel._version 7.1.0
json 2.0.9
jupyter_client._version 8.6.3
platform 1.0.8
zmq.sugar.version 27.1.0
zmq.sugar 27.1.0
zmq 27.1.0
logging 0.5.1.2
traitlets._version 5.14.3
traitlets 5.14.3
jupyter_core.version 5.9.1
jupyter_core 5.9.1
tornado 6.5.2
zlib 1.0
_ctypes 1.1.0
ctypes 1.1.0
colorama 0.4.6
_curses b'2.2'
socketserver 0.4
argparse 1.1
dateutil._version 2.9.0.post0
dateutil 2.9.0.post0
six 1.17.0
_decimal 1.70
decimal 1.70
platformdirs.version 4.5.0
platformdirs 4.5.0
_csv 1.0
csv 1.0
jupyter_client 8.6.3
ipykernel 7.1.0
IPython.core.release 8.37.0
executing.version 2.2.1
executing 2.2.1
pure_eval.version 0.2.3
pure_eval 0.2.3
stack_data.version 0.6.3
stack_data 0.6.3
pygments 2.19.2
IPython.core.crashhandler 8.37.0
pickleshare 0.7.5
decorator 5.2.1
_sqlite3 2.6.0
sqlite3.dbapi2 2.6.0
sqlite3 2.6.0
exceptiongroup._version 1.3.0
exceptiongroup 1.3.0
wcwidth 0.2.14
prompt_

/tmp/ipykernel_1509913/1799866556.py:4: DeprecationWarning: numpy.core is deprecated and has been renamed to numpy._core. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.__version__.
  print(module,sys.modules[module].__version__)
/tmp/ipykernel_1509913/1799866556.py:4: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print(module,sys.modules[module].__version__)


attr 25.4.0
lightning.pytorch 2.5.6
lightning 2.5.6
pyro._version 1.9.1
pyro 1.9.1
absl 2.3.1
ml_collections 1.1.0
docrep 0.3.2
jax.version 0.6.2
jaxlib.version 0.6.2
jaxlib 0.6.2
ml_dtypes 0.5.4
etils 1.13.0
filelock.version 3.20.0
filelock 3.20.0
jax.lib 0.6.2
jax 0.6.2
mudata._version 0.3.2
mudata 0.3.2
msgpack 1.1.2
flax.version 0.10.7
flax 0.10.7
xml.sax.handler 2.0beta
toolz 1.1.0
chex 0.1.90
optax 0.2.6
multipledispatch 0.6.0
numpyro.version 0.19.0
numpyro 0.19.0
sparse._version 0.17.0
sparse 0.17.0
xarray.tutorial master
xarray 2025.6.1
scvi 1.3.3
cffi 2.0.0
_cffi_backend 2.0.0
pycparser.ply 3.9
pycparser.ply.yacc 3.10
pycparser.ply.lex 3.10
pycparser 2.22
pynndescent 0.5.13
umap 0.5.9.post2
cell2location 0.1.5
